# P07 Wan2.2 Animate — 分步代码执行

本 Notebook 从 Python 代码层逐步跑通 **P07 动作迁移** 全流程（不经过 6020 Web 页面）。

## 端口对照

| 端口 | 服务 | 用途 |
|------|------|------|
| **6006** | ComfyUI 原生 Web UI | **节点画布**、Load 工作流、Queue Prompt |
| **6020** | wan-animate-api + 莫兰迪前端 | 项目封装的一键生成页 |
| 本 Notebook | HTTP `/prompt` | 与 CLI/API 相同，直连 ComfyUI **6006** |

在 ComfyUI 画布中手动跑工作流：打开 `http://<主机>:6006`，Load UI 格式 JSON（见 [`notebooks/README.md`](README.md)）。

资产清单：[`docs/ASSETS_MIGRATION.md`](../docs/ASSETS_MIGRATION.md)

In [1]:
# Step 1 — 环境与配置
import json
import sys
import time
from pathlib import Path

ROOT = Path("/root/zfhs-wan-animate")
if not ROOT.is_dir():
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from zfhs_wan_animate.runner import load_config, resolve_runtime

cfg = load_config()
rt = resolve_runtime(cfg)

# --- 用户覆盖（填路径则优先；留 None 则用 config samples）---
USER_IMAGE: str | None = None   # 例: "/root/ComfyUI/input/wonyoung.jpg"
USER_VIDEO: str | None = None   # 例: "/root/ComfyUI/input/AI小红狗.mp4"

samples = cfg.get("samples", {})
IMAGE_PATH = Path(USER_IMAGE or samples.get("image", "/root/ComfyUI/input/image (17).png"))
VIDEO_PATH = Path(USER_VIDEO or samples.get("video", ""))

WIDTH, HEIGHT, SECONDS = 468, 832, 5  # 默认 5 秒试跑
FPS = int(rt["fps"])
FRAMES = max(1, SECONDS * FPS)

print("项目根目录:", ROOT)
print("ComfyUI URL:", rt["url"])
print("ComfyUI root:", rt["comfy_root"])
print("工作流:", rt["workflow_path"])
print(f"参数: {WIDTH}x{HEIGHT}, {SECONDS}s -> {FRAMES} 帧")
print("角色图:", IMAGE_PATH)
print("动作视频:", VIDEO_PATH)

项目根目录: /root/zfhs-wan-animate
ComfyUI URL: http://127.0.0.1:6006
ComfyUI root: /root/ComfyUI
工作流: /root/zfhs-wan-animate/workflows/p07_animate_v4.json
参数: 468x832, 5s -> 150 帧
角色图: /root/ComfyUI/input/image (17).png
动作视频: /root/ComfyUI/input/5053929f1d2c2ef117a3a8b8c02075c7da53e5380365bc2f8a87992986058e39.mp4


In [ ]:
# Step 1b — 工作流预设与高级调参
from zfhs_wan_animate.workflow_p07 import extract_tunable_defaults

WORKFLOW_VARIANT = "v5"  # "v4" 标准动作迁移 | "v5" 保身份动作迁移

# 留空 {} 则使用该 variant JSON 内默认值；可覆盖下方键
TUNABLES: dict = {
    # 身份与姿态
    # "62:pose_strength": 0.65,
    # "62:face_strength": 0.2,
    # "996:draw_head": False,
    # "64:crop_position": "center",
    # 采样
    # "27:steps": 4,
    # "27:denoise_strength": 1.0,
    # LoRA（0 = 关闭）
    # "171:strength_0": 0.3,
    # "171:strength_2": 0,
    # 提示词
    # "65:positive_prompt": "保持参考图人物身份...",
    # "65:negative_prompt": "anime face, distorted proportions...",
}

variant_cfg = cfg["workflows"][WORKFLOW_VARIANT]
WORKFLOW_PATH = ROOT / variant_cfg["file"]
rt = resolve_runtime(cfg, workflow_path=WORKFLOW_PATH)

wf_defaults = extract_tunable_defaults(WORKFLOW_PATH)
MERGED_TUNABLES = {**wf_defaults, **TUNABLES}

print("工作流预设:", WORKFLOW_VARIANT, "-", variant_cfg.get("label"))
print("工作流文件:", WORKFLOW_PATH)
print("调参:", json.dumps(MERGED_TUNABLES, ensure_ascii=False, indent=2))

In [ ]:
# Step 2 — 资产自检（模型缺失仅警告，不阻断）
import subprocess

result = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "verify_assets.py")],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("WARN: 部分资产缺失，生成可能失败。详见 docs/ASSETS_MIGRATION.md")

In [ ]:
# Step 3 — ComfyUI 连通性
from zfhs_wan_animate.comfy_client import ComfyClient

client = ComfyClient(rt["url"], comfy_root=rt["comfy_root"])
stats = client.check_ready()
sys_info = stats.get("system", {})
print("ComfyUI 版本:", sys_info.get("comfyui_version"))
print("RAM 可用 (GB):", round(sys_info.get("ram_free", 0) / 1024**3, 2))
devices = stats.get("devices", [])
if devices:
    print("GPU:", devices[0].get("name"), "| VRAM free:", devices[0].get("vram_free"))

In [ ]:
# Step 4 — 上传素材（节点 57 角色图、997 动作视频）
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f"角色图不存在: {IMAGE_PATH}")
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f"动作视频不存在: {VIDEO_PATH}")

image_name = client.upload_image(IMAGE_PATH)
video_name = client.upload_video(VIDEO_PATH)
print("节点 57  image:", image_name)
print("节点 997 video:", video_name)

In [ ]:
# Step 5 — 构建 API Prompt（含工作流 variant + 调参）
from zfhs_wan_animate.workflow_p07 import build_prompt_with_tunables

prompt = build_prompt_with_tunables(
    image_name=image_name,
    video_name=video_name,
    width=WIDTH,
    height=HEIGHT,
    seconds=SECONDS,
    workflow_path=WORKFLOW_PATH,
    fps=FPS,
    tunables=MERGED_TUNABLES,
)

print("节点 62 pose_strength:", prompt["62"]["inputs"]["pose_strength"])
print("节点 62 face_strength:", prompt["62"]["inputs"]["face_strength"])
print("节点 1001 宽:", prompt["1001"]["inputs"]["value"])
print("节点 1002 高:", prompt["1002"]["inputs"]["value"])
print("节点 1003 帧:", prompt["1003"]["inputs"]["value"])
print("节点 867 音轨:", prompt["867"]["inputs"].get("audio"))
print("\nPrompt 预览（前 2000 字符）:")
print(json.dumps(prompt, indent=2, ensure_ascii=False)[:2000], "...")

In [ ]:
# Step 6 — 提交任务
import uuid

client_id = uuid.uuid4().hex
prompt_id = client.queue_prompt(prompt, client_id=client_id)

queue = client.get_queue()
running = len(queue.get("queue_running", []))
pending = len(queue.get("queue_pending", []))

print("prompt_id:", prompt_id)
print("client_id:", client_id)
print(f"队列: running={running}, pending={pending}")

In [ ]:
# Step 7 — 轮询进度（WanVideo Sampler 节点 27 较慢，请耐心等待）
from zfhs_wan_animate.runner import poll_p07
from zfhs_wan_animate.comfy_errors import parse_comfy_execution_error

POLL_INTERVAL = 3
TIMEOUT_SEC = max(20 * 60, SECONDS * 45 + 8 * 60)
start = time.monotonic()
polled = None

while time.monotonic() - start < TIMEOUT_SEC:
    polled = poll_p07(prompt_id, config=cfg, ref_video_path=VIDEO_PATH)
    if not polled.pending:
        break
    elapsed = int(time.monotonic() - start)
    print(f"等待中... {elapsed}s / 超时 {TIMEOUT_SEC}s")
    time.sleep(POLL_INTERVAL)

if polled is None or polled.pending:
    raise TimeoutError(f"轮询超时（{TIMEOUT_SEC}s）。任务可能仍在 ComfyUI 队列中，prompt_id={prompt_id}")

if polled.error:
    print("生成失败:", polled.error)
    if polled.history:
        msgs = polled.history.get("status", {}).get("messages", [])
        print("原始错误:", parse_comfy_execution_error(msgs))
    raise RuntimeError(polled.error)

print(f"完成，耗时 {int(time.monotonic() - start)}s，输出 {len(polled.outputs)} 个文件")

In [ ]:
# Step 8 — 获取结果与音轨
for i, out in enumerate(polled.outputs):
    local = out.local_path(rt["comfy_root"])
    print(f"--- output[{i}] ---")
    print("  本地路径:", local)
    print("  预览 URL:", out.view_url)
    print("  存在:", local.is_file())

primary = polled.outputs[0] if polled.outputs else None
OUTPUT_PATH = primary.local_path(rt["comfy_root"]) if primary else None

In [ ]:
# Step 9 — 预览输出视频
from IPython.display import Video, display

if OUTPUT_PATH and OUTPUT_PATH.is_file():
    display(Video(str(OUTPUT_PATH), embed=True, width=480))
else:
    print("未找到本地视频文件，可访问 view URL:", primary.view_url if primary else "N/A")

## 附录

### 在 ComfyUI 画布（6006）中手动执行

1. 确保 ComfyUI 运行在 **6006**（`bash scripts/start-comfyui.sh`）
2. 浏览器打开 `http://<主机IP>:6006`
3. **Load** UI 格式工作流：`assets/workflows/ui/p07_animate_v4_ui.json`
4. 设置节点 57 / 997 / 1001 / 1002 / 1003，点击 **Queue Prompt**

> API Prompt 格式（本 Notebook 使用）见 `workflows/p07_animate_v4.json`，与画布 UI JSON 不同。

### 中断当前任务

运行下方 cell 可调用 `interrupt_comfy()`。

In [ ]:
# 附录 — 中断 / 历史 / 一键对照
from zfhs_wan_animate.runner import interrupt_comfy, run_p07

# 取消注释以中断 ComfyUI 当前任务：
# interrupt_comfy(config=cfg)
# print("已发送 interrupt")

# 查看 history 原始消息：
# hist = client.get_history_entry(prompt_id)
# print(json.dumps(hist.get("status", {}), indent=2)[:3000])

# 一键版对照（等价于 scripts/run_p07.py）：
# one_shot = run_p07(
#     image_path=IMAGE_PATH,
#     video_path=VIDEO_PATH,
#     width=WIDTH, height=HEIGHT, seconds=SECONDS, config=cfg,
# )
# print(one_shot.prompt_id, one_shot.elapsed_seconds)